# Part 1 · Multicluster ambient

**The story for the customer:** run the same app in two clusters, give it one hostname, and show traffic quietly fail over to the other cluster when one goes down, with no change to the app.

`setup.sh` has already installed Solo ambient on both clusters, sharing a single root CA, so they trust each other. From here we deploy the app, join the clusters, then break things on purpose to prove the resilience.

**Cluster layout (matches the deck):**

- `mesh1` runs reviews **v1 + v2** (no stars / black stars)
- `mesh2` also runs **v3** (red stars)

So the moment traffic crosses to mesh2, you can point at the red stars on screen.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="640" height="250" rx="10" fill="#f8fafc"/> <text x="320" y="30" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Two kind clusters, ambient mesh, shared root of trust</text> <!-- mesh1 --> <rect x="30" y="55" width="250" height="150" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="155" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1  (kind)</text> <rect x="55" y="95" width="200" height="34" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="155" y="116" text-anchor="middle" font-size="12" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="55" y="140" width="200" height="30" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="155" y="159" text-anchor="middle" font-size="11.5" fill="#334155">trust domain: mesh1</text> <text x="155" y="190" text-anchor="middle" font-size="11" fill="#64748b">no workloads yet</text> <!-- mesh2 --> <rect x="360" y="55" width="250" height="150" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="485" y="80" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2  (kind)</text> <rect x="385" y="95" width="200" height="34" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="485" y="116" text-anchor="middle" font-size="12" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="385" y="140" width="200" height="30" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="485" y="159" text-anchor="middle" font-size="11.5" fill="#334155">trust domain: mesh2</text> <text x="485" y="190" text-anchor="middle" font-size="11" fill="#64748b">no workloads yet</text> <!-- shared root CA between them --> <rect x="292" y="112" width="56" height="36" rx="6" fill="#fef9c3" stroke="#ca8a04"/> <text x="320" y="128" text-anchor="middle" font-size="10" fill="#713f12">root</text> <text x="320" y="141" text-anchor="middle" font-size="10" fill="#713f12">CA</text> <line x1="280" y1="130" x2="292" y2="130" stroke="#ca8a04" stroke-width="1.5"/> <line x1="348" y1="130" x2="360" y2="130" stroke="#ca8a04" stroke-width="1.5"/> <text x="320" y="228" text-anchor="middle" font-size="11" fill="#64748b">Same root CA signs each cluster's intermediate; one root of trust, not peered yet.</text> </svg></div>

> **Kernel:** Bash (Select Kernel → Jupyter Kernel → **Bash**).
> This notebook is **self-contained**: run **Connect**, then **Reset**, then the steps top to bottom.
> It needs `./setup.sh` to have been run once. After a laptop sleep: `./demo-scripts/wake.sh`.

### Connect

Sets the contexts, the Solo `istioctl` build and your licence env, and confirms the platform is up. **Run this first, always.**

In [ ]:
# Connect — contexts, the Solo istioctl build, licences. Run this first, always.
[ -d istio-ambient-demo-kind ] && cd istio-ambient-demo-kind || :
export CLUSTER1=kind-mesh1 CLUSTER2=kind-mesh2
export ISTIOCTL=$HOME/.istioctl/bin/istioctl-1.30.3-solo
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "mesh1=$CLUSTER1  mesh2=$CLUSTER2  licence: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"
$ISTIOCTL --context $CLUSTER1 multicluster check 2>&1 | grep -E "Peers Check|Gateway Check" || echo "not peered — run ./setup.sh (or ./demo-scripts/wake.sh after a sleep)"

## Reset · run this to (re)start the demo

Clears anything a previous run of this demo left behind (waypoint, canary, rate limit, the global/takeover labels) and puts `reviews` back to two pods. Safe on a fresh cluster: every command is a no-op when there is nothing to remove.

In [ ]:
# Reset — clean slate for this demo. Safe to run on a fresh cluster (all no-ops).
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}"
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint- 2>/dev/null || true
kubectl --context $CLUSTER1 -n bookinfo delete httproute reviews-mesh-split reviews-canary --ignore-not-found 2>/dev/null || true
kubectl --context $CLUSTER1 -n bookinfo delete gateway bookinfo-waypoint --ignore-not-found 2>/dev/null || true
kubectl --context $CLUSTER1 -n bookinfo delete agentgatewaypolicy ingress-ratelimit --ignore-not-found 2>/dev/null || true
for C in $CLUSTER1 $CLUSTER2; do
  kubectl --context $C -n bookinfo label svc reviews solo.io/service-scope- solo.io/service-takeover- 2>/dev/null || true
done
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1 2>/dev/null || true; done
echo "  ✓ clean — §1.1 below (re)deploys bookinfo; every step is idempotent"

## 1.1 · Deploy the app and put it in the mesh

**What we're doing:** deploy Bookinfo into both clusters and enrol it into the ambient mesh.

**How:** two labels on the namespace, no sidecars and no restarts:

- `istio.io/dataplane-mode=ambient`: ztunnel starts capturing every pod's traffic
- `topology.istio.io/network=<cluster>`: tells istiod which cluster the pods live in, so it can route across clusters later

**What you'll see:** each namespace carrying both labels, then the deployments per cluster, mesh1 with reviews v1 + v2 and mesh2 also with v3.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX create namespace bookinfo --dry-run=client -o yaml | kubectl --context $CTX apply -f -
  kubectl --context $CTX label namespace bookinfo \
    istio.io/dataplane-mode=ambient \
    topology.istio.io/network=${CTX#kind-} --overwrite
done

# ── verify: both namespaces carry the ambient + network labels ──
echo "mesh1:"; kubectl --context $CLUSTER1 get ns bookinfo -L istio.io/dataplane-mode -L topology.istio.io/network
echo "mesh2:"; kubectl --context $CLUSTER2 get ns bookinfo -L istio.io/dataplane-mode -L topology.istio.io/network

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
BOOK=https://raw.githubusercontent.com/istio/istio/release-1.24/samples/bookinfo/platform/kube
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo.yaml
  kubectl --context $CTX -n bookinfo apply -f $BOOK/bookinfo-versions.yaml
done
# match the deck: mesh1 runs v1+v2 only; mesh2 keeps v3 (red stars) as well
kubectl --context $CLUSTER1 -n bookinfo delete deploy reviews-v3 --ignore-not-found

# a client pod on mesh1. It does DOUBLE duty:
#   1. a steady ~1 req/s background loop to the GLOBAL hostname, so the Gloo UI
#      service graph continuously reflects routing (mesh1 normally, shifts to
#      mesh2 during failover, back on restore — visible live, no manual traffic).
#      It is a harmless no-op until §1.4 creates reviews.bookinfo.mesh.internal.
#   2. the measurement pod: the observe cells below `kubectl exec` their own
#      curls into it for the precise podname counts (exec spawns an independent
#      process, so the loop and the measurements never interfere).
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: apps/v1
kind: Deployment
metadata: { name: demo-client, namespace: bookinfo, labels: { app: demo-client } }
spec:
  replicas: 1
  selector: { matchLabels: { app: demo-client } }
  template:
    metadata: { labels: { app: demo-client } }
    spec:
      containers:
        - name: curl
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                curl -s -o /dev/null -m 4 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null || true
                sleep 1
              done
EOF

kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s
kubectl --context $CLUSTER2 -n bookinfo wait --for=condition=Ready pod --all --timeout=240s

# ── verify: mesh1 has reviews v1+v2 (no v3); mesh2 has v1+v2+v3 ──
echo "mesh1 deployments:"; kubectl --context $CLUSTER1 -n bookinfo get deploy
echo "mesh2 deployments:"; kubectl --context $CLUSTER2 -n bookinfo get deploy

## 1.2 · Join the two clusters into one mesh

**What we're doing:** peer the clusters so a service in one can be reached from the other.

**How:** two istioctl commands (the Solo build).

`expose` builds the east-west gateway on each cluster. It sits on a LoadBalancer IP with an HBONE listener on `:15008` (the cross-cluster mTLS tunnel real traffic flows through) and istiod's xDS on `:15012`.

`link` then points each cluster at the other's east-west address, so the two istiods discover each other's endpoints. No remote kubeconfig secrets, Solo peers istiod-to-istiod over xDS.

`setup.sh` already ran these, so they no-op here. Run them anyway to show how the clusters were joined.

**What you'll see:** on each cluster, two Gateways, the local `istio-eastwest` plus a `istio-remote-peer-<other>` pointing at the peer, and `multicluster check` reporting the clusters connected.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# east-west gateway on each cluster (idempotent)
for CTX in $CLUSTER1 $CLUSTER2; do
  $ISTIOCTL --context $CTX multicluster expose -n istio-eastwest
done
# link them bidirectionally (idempotent)
$ISTIOCTL multicluster link --namespace istio-eastwest --contexts=$CLUSTER1,$CLUSTER2

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# what link created: the local east-west gateway (istio-eastwest) + a remote
# peer gateway (istio-remote-peer-<other>) per cluster. Class + address make the
# local-vs-remote distinction obvious.
for C in $CLUSTER1 $CLUSTER2; do
  echo "── ${C#kind-} ──"
  kubectl --context $C -n istio-eastwest get gateways \
    -o custom-columns=NAME:.metadata.name,CLASS:.spec.gatewayClassName,ADDRESS:.status.addresses[0].value
done
echo
$ISTIOCTL --context $CLUSTER1 multicluster check | grep -E "Peers Check|Gateway Check"

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 290" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="640" height="290" rx="10" fill="#f8fafc"/> <text x="320" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Peered over HBONE: east-west gateways + link (bidirectional)</text> <defs> <marker id="ar2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto-start-reverse"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- mesh1 --> <rect x="20" y="50" width="240" height="170" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="140" y="74" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1</text> <rect x="42" y="88" width="196" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="140" y="108" text-anchor="middle" font-size="11.5" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="42" y="128" width="196" height="40" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="140" y="145" text-anchor="middle" font-size="12" font-weight="600" fill="#7c2d12">east-west GW (LoadBalancer)</text> <text x="140" y="160" text-anchor="middle" font-size="11" fill="#92400e">192.168.97.142</text> <rect x="42" y="178" width="196" height="28" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="140" y="196" text-anchor="middle" font-size="10.5" fill="#475569">knows istio-remote-peer-mesh2</text> <!-- mesh2 --> <rect x="380" y="50" width="240" height="170" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="500" y="74" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2</text> <rect x="402" y="88" width="196" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="500" y="108" text-anchor="middle" font-size="11.5" fill="#1e293b">istiod + ztunnel (ambient)</text> <rect x="402" y="128" width="196" height="40" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="500" y="145" text-anchor="middle" font-size="12" font-weight="600" fill="#7c2d12">east-west GW (LoadBalancer)</text> <text x="500" y="160" text-anchor="middle" font-size="11" fill="#92400e">192.168.97.160</text> <rect x="402" y="178" width="196" height="28" rx="6" fill="#ffffff" stroke="#93c5fd"/> <text x="500" y="196" text-anchor="middle" font-size="10.5" fill="#475569">knows istio-remote-peer-mesh1</text> <!-- BIDIRECTIONAL arrows between the two east-west GWs (in the gap, wall to wall) --> <line x1="260" y1="134" x2="380" y2="134" stroke="#334155" stroke-width="1.6" marker-start="url(#ar2)" marker-end="url(#ar2)"/> <text x="320" y="126" text-anchor="middle" font-size="11" font-weight="600" fill="#0f172a">HBONE :15008</text> <text x="320" y="151" text-anchor="middle" font-size="10" fill="#475569">cross-cluster mTLS tunnel</text> <line x1="260" y1="186" x2="380" y2="186" stroke="#64748b" stroke-width="1.4" stroke-dasharray="4 3" marker-start="url(#ar2)" marker-end="url(#ar2)"/> <text x="320" y="180" text-anchor="middle" font-size="11" font-weight="600" fill="#0f172a">xDS :15012</text> <text x="320" y="205" text-anchor="middle" font-size="10" fill="#475569">endpoint discovery</text> <text x="320" y="255" text-anchor="middle" font-size="11" fill="#64748b">expose builds each local east-west GW; link points each cluster at the peer's address.</text> <text x="320" y="272" text-anchor="middle" font-size="11" fill="#64748b">Either cluster can reach the other's services; no remote kubeconfig secrets.</text> </svg></div>

<details>
<summary><strong>Optional (click to expand): the GitOps way, the same peering as two declarative Gateways</strong></summary>

`istioctl multicluster link` is a convenience wrapper. Under the hood it creates one `Gateway` (class `istio-remote`) per cluster in the `istio-eastwest` namespace, each pointing at the peer's east-west address. If you drive the mesh from Git rather than a CLI, you apply those Gateways yourself with `kubectl`. Below are the exact resources `link` created on this demo, so the alternative is a plain `kubectl apply` in each cluster.

```yaml
# apply to mesh1, points at mesh2's east-west gateway
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: istio-remote-peer-mesh2
  namespace: istio-eastwest
  annotations:
    gateway.istio.io/service-account: istio-eastwest
    gateway.istio.io/trust-domain: mesh2
    peering.solo.io/preferred-data-plane-service-type: loadbalancer
  labels:
    topology.istio.io/cluster: mesh2
    topology.istio.io/network: mesh2
spec:
  gatewayClassName: istio-remote
  addresses:
  - type: IPAddress
    value: 192.168.97.160          # mesh2's east-west LoadBalancer IP (environment-specific)
  listeners:
  - name: xds-tls
    port: 15012
    protocol: TLS
    tls: { mode: Passthrough }
    allowedRoutes: { namespaces: { from: Same } }
```

```yaml
# apply to mesh2, points at mesh1's east-west gateway
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: istio-remote-peer-mesh1
  namespace: istio-eastwest
  annotations:
    gateway.istio.io/service-account: istio-eastwest
    gateway.istio.io/trust-domain: mesh1
    peering.solo.io/preferred-data-plane-service-type: loadbalancer
  labels:
    topology.istio.io/cluster: mesh1
    topology.istio.io/network: mesh1
spec:
  gatewayClassName: istio-remote
  addresses:
  - type: IPAddress
    value: 192.168.97.142          # mesh1's east-west LoadBalancer IP (environment-specific)
  listeners:
  - name: xds-tls
    port: 15012
    protocol: TLS
    tls: { mode: Passthrough }
    allowedRoutes: { namespaces: { from: Same } }
```

A few things to notice, because they are what make this environment-specific rather than copy-paste:

- **The address is the peer's east-west LoadBalancer IP**, so it is not portable. `link` reads it for you; declaratively you template it (Argo, Helm) or patch it once the LoadBalancer has an address.
- **`trust-domain`, `topology.istio.io/cluster` and `network` are each cluster's own name here (`mesh1` / `mesh2`), not `cluster.local`.** They must match how each cluster was installed, or the peer's identities will not validate.
- The east-west gateway itself (what `istioctl multicluster expose` builds) is likewise just a `Gateway`, of class `istio-eastwest`. On this demo `setup.sh` already created it, so `expose` is a no-op.

</details>

## 1.3 · Put the app behind an agentgateway ingress

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 660 300" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="660" height="300" rx="10" fill="#f8fafc"/> <text x="330" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">agentgateway ingress fronts the app on mesh1</text> <defs> <marker id="ar3" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker> </defs> <!-- external client --> <rect x="15" y="120" width="70" height="50" rx="8" fill="#e2e8f0" stroke="#64748b"/> <text x="50" y="141" text-anchor="middle" font-size="11" fill="#334155">client /</text> <text x="50" y="156" text-anchor="middle" font-size="11" fill="#334155">browser</text> <line x1="85" y1="145" x2="108" y2="145" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <!-- mesh1 --> <rect x="110" y="48" width="240" height="200" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="230" y="70" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh1</text> <rect x="128" y="82" width="204" height="42" rx="6" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="230" y="100" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">agentgateway ingress (LB)</text> <text x="230" y="115" text-anchor="middle" font-size="10.5" fill="#166534">class: enterprise-agentgateway</text> <line x1="230" y1="124" x2="230" y2="139" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <rect x="138" y="140" width="184" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="230" y="160" text-anchor="middle" font-size="12" fill="#1e293b">productpage / reviews …</text> <rect x="150" y="182" width="180" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="240" y="199" text-anchor="middle" font-size="11" font-weight="600" fill="#7c2d12">east-west GW</text> <text x="240" y="211" text-anchor="middle" font-size="10" fill="#92400e">.142</text> <!-- mesh2 --> <rect x="470" y="48" width="175" height="200" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/> <text x="557" y="70" text-anchor="middle" font-size="14" font-weight="700" fill="#1e3a8a">mesh2</text> <rect x="486" y="140" width="145" height="30" rx="6" fill="#dbeafe" stroke="#60a5fa"/> <text x="558" y="160" text-anchor="middle" font-size="12" fill="#1e293b">reviews (+v3) …</text> <rect x="486" y="182" width="145" height="34" rx="6" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/> <text x="558" y="199" text-anchor="middle" font-size="11" font-weight="600" fill="#7c2d12">east-west GW</text> <text x="558" y="211" text-anchor="middle" font-size="10" fill="#92400e">.160</text> <!-- HBONE arrow ENTIRELY in the gap: mesh1 wall (350) to mesh2 wall (470) --> <text x="410" y="184" text-anchor="middle" font-size="10.5" font-weight="600" fill="#0f172a">HBONE :15008</text> <line x1="350" y1="199" x2="470" y2="199" stroke="#334155" stroke-width="1.6" marker-end="url(#ar3)"/> <text x="330" y="280" text-anchor="middle" font-size="11" fill="#64748b">One north-south front door on mesh1; the clusters stay peered over HBONE for the failover steps.</text> </svg></div>

**What we're doing:** give the app a front door on mesh1, then open it in the browser.

**How:** one `Gateway` (class `enterprise-agentgateway`, Solo's standard Gateway API data plane) and one `HTTPRoute` for productpage. It lands on a LoadBalancer IP.

**What you'll see:** the ingress address printed, an HTTP 200 from `/productpage`, and the page in your browser.

**Keep that browser tab open.** In §1.5 to §1.6 you'll watch the reviews stars break and come back, red meaning served from mesh2.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: bookinfo-gateway, namespace: bookinfo }
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - { name: http, port: 8080, protocol: HTTP, allowedRoutes: { namespaces: { from: Same } } }
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: productpage, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - backendRefs: [{ name: productpage, port: 9080 }]
EOF
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Programmed gateway/bookinfo-gateway --timeout=90s
# the agentgateway data-plane pod is created ON DEMAND, a few seconds AFTER the
# Gateway goes Programmed — poll for it before waiting Ready (kubectl wait errors
# on "no matching resources" if the pod does not exist yet)
for i in $(seq 1 40); do
  kubectl --context $CLUSTER1 -n bookinfo get pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway 2>/dev/null | grep -q . && break
  sleep 3
done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready \
  pod -l gateway.networking.k8s.io/gateway-name=bookinfo-gateway --timeout=120s

GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo "open →  http://$GW:8080/productpage"
# LB IP + pod may need a moment to answer — retry until the first 200
for i in $(seq 1 20); do
  code=$(curl -s -o /dev/null -w '%{http_code}' -m 5 "http://$GW:8080/productpage")
  [ "$code" = "200" ] && break; sleep 3
done
echo "HTTP $code"

## 1.4 · Give the service one hostname across both clusters

**What we're doing:** make `reviews` reachable on a single name that works no matter which cluster is actually serving it. This is the foundation for the failover we prove next.

Right now our sample client (`demo-client`) is already calling `reviews.bookinfo.mesh.internal` once a second, and has been since §1.1. That name is a *global* hostname, and it does not exist yet, so every call quietly fails and the Gloo UI graph stays empty. This step is what creates it. The moment we publish the global hostname, the client's calls land and the graph comes alive.

**How:** one label, `solo.io/service-scope=global`. Solo Istio publishes `reviews.bookinfo.mesh.internal` as a ServiceEntry whose endpoints span both clusters, and it leaves the original `reviews.bookinfo.svc.cluster.local` local-only, so nothing changes for callers already using it.

By default this global hostname prefers **local** endpoints and only crosses to the other cluster when the local ones are gone. That preference is what makes the next two steps a clean failover.

**What you'll see:** the ServiceEntry istiod generated for the global hostname, and every call from mesh1 still served by mesh1's own pods, because local is healthy.

Open the Gloo UI Graph now.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for CTX in $CLUSTER1 $CLUSTER2; do
  kubectl --context $CTX -n bookinfo label svc reviews solo.io/service-scope=global --overwrite
done
sleep 8

# the label makes istiod publish the global hostname as a ServiceEntry, shown here:
echo "  == global hostname istiod published (ServiceEntry) =="
kubectl --context $CLUSTER1 get serviceentry -A \
  -o custom-columns='NAMESPACE:.metadata.namespace,NAME:.metadata.name,HOSTS:.spec.hosts,LOCATION:.spec.location,RESOLUTION:.spec.resolution' \
  | grep -E 'NAMESPACE|mesh\.internal'
echo
$ISTIOCTL --context $CLUSTER1 multicluster check 2>/dev/null | grep -iA2 "shared services" || true

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# mesh1 still has local reviews pods, so calls to the GLOBAL hostname stay LOCAL
echo "  ══ 10 calls to the GLOBAL hostname from mesh1  ·  reviews.bookinfo.mesh.internal ══"
for i in $(seq 1 10); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]-[a-z0-9]+-[a-z0-9]+'
done | sort | uniq -c | awk '{printf "    %-34s ×%d\n", $2, $1}'
echo "    all served by mesh1 pods  ·  traffic stays local while local is healthy"

<details>
<summary><strong>What got created (click to expand): the ServiceEntry behind the global hostname</strong></summary>

`solo.io/service-scope=global` is not a manifest you wrote. istiod autogenerates a **ServiceEntry** in `istio-system`, named `autogen.<namespace>.<service>`, that publishes the global hostname with endpoints spanning both clusters. This is the object on this demo:

```yaml
apiVersion: networking.istio.io/v1
kind: ServiceEntry
metadata:
  name: autogen.bookinfo.reviews          # namespace: istio-system
  labels:
    solo.io/parent-service: reviews
    solo.io/parent-service-namespace: bookinfo
    solo.io/service-scope: global
    admin.solo.io/segment: default
  annotations:
    networking.istio.io/traffic-distribution: PreferNetwork   # prefer local, cross only when local is gone
spec:
  hosts:
  - reviews.bookinfo.mesh.internal
  location: MESH_INTERNAL
  resolution: STATIC
  ports:
  - { name: http, number: 9080, protocol: HTTP, targetPort: 9080 }
  subjectAltNames:
  - spiffe://mesh2/ns/bookinfo/sa/bookinfo-reviews     # accepts mesh2's workload identity
  workloadSelector:
    labels: { app: reviews }
status:
  addresses:
  - { host: reviews.bookinfo.mesh.internal, value: 240.240.0.4 }   # synthetic VIP for the global name
```

`PreferNetwork` is what keeps the global name on local pods while they exist, then fails it over. The `subjectAltNames` line is why the caller trusts a reviews pod running under mesh2's identity.

See it live:
```
kubectl --context kind-mesh1 -n istio-system get serviceentry autogen.bookinfo.reviews -o yaml
```

</details>

## 1.5 · Kill it locally and watch it fail over

**What we're doing:** take mesh1's `reviews` away completely and show the global hostname keep serving from mesh2.

**How:** scale mesh1's reviews to zero. Nothing else changes, same hostname, same callers.

**What you'll see, two hostnames diverging:**

- the **global** hostname (`...mesh.internal`) keeps answering, now from mesh2's pods including v3, over the HBONE tunnel
- the **local** hostname (`...svc.cluster.local`) breaks, because §1.4 left it local-only and there is nothing local left

In the Gloo UI Graph the demo-client edge crosses from mesh1 to mesh2. In the browser, productpage's reviews panel goes down (it uses the local hostname). §1.6 fixes exactly that.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=0; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=delete pod -l app=reviews --timeout=60s
sleep 8

echo "  ══ GLOBAL hostname  ·  still serving, now from mesh2  (v3 = mesh2) ══"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.mesh.internal:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]-[a-z0-9]+-[a-z0-9]+'
done | sort | uniq -c | awk '{printf "    %-34s ×%d\n", $2, $1}'

echo
echo "  ══ LOCAL hostname  ·  local-only, and local is gone ══"
code=$(kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
  curl -s -o /dev/null -w '%{http_code}' -m 5 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null || echo 000)
echo "    reviews.bookinfo.svc.cluster.local  →  $code   (broken: nothing local left)"

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Kill mesh1's reviews: the global hostname fails over, the local one breaks</text> <defs> <marker id="g" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> <marker id="r" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#dc2626"/></marker> </defs> <!-- LANE 1: global hostname, still served from mesh2 --> <rect x="20" y="58" width="120" height="38" rx="7" fill="#eef2ff" stroke="#6366f1"/> <text x="80" y="82" text-anchor="middle" font-size="11.5" fill="#312e81">caller (mesh1)</text> <line x1="140" y1="77" x2="198" y2="77" stroke="#16a34a" stroke-width="2" marker-end="url(#g)"/> <rect x="198" y="58" width="222" height="38" rx="7" fill="#dbeafe" stroke="#60a5fa"/> <text x="309" y="82" text-anchor="middle" font-size="11" fill="#1e293b">reviews.bookinfo.mesh.internal</text> <text x="448" y="70" text-anchor="middle" font-size="9.5" fill="#166534">HBONE</text> <line x1="420" y1="77" x2="478" y2="77" stroke="#16a34a" stroke-width="2" marker-end="url(#g)"/> <rect x="478" y="58" width="222" height="38" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="589" y="82" text-anchor="middle" font-size="11.5" font-weight="600" fill="#14532d">mesh2: reviews v3  ✓</text> <text x="360" y="116" text-anchor="middle" font-size="11" fill="#166534">Global hostname: still served, now from mesh2 (red stars).</text> <!-- LANE 2: local hostname, dead --> <rect x="20" y="150" width="120" height="38" rx="7" fill="#eef2ff" stroke="#6366f1"/> <text x="80" y="174" text-anchor="middle" font-size="11.5" fill="#312e81">caller (mesh1)</text> <line x1="140" y1="169" x2="198" y2="169" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#r)"/> <rect x="198" y="150" width="222" height="38" rx="7" fill="#dbeafe" stroke="#60a5fa"/> <text x="309" y="174" text-anchor="middle" font-size="11" fill="#1e293b">reviews.bookinfo.svc.cluster.local</text> <line x1="420" y1="169" x2="478" y2="169" stroke="#dc2626" stroke-width="2" stroke-dasharray="5 3" marker-end="url(#r)"/> <text x="449" y="158" text-anchor="middle" font-size="13" font-weight="700" fill="#dc2626">✗</text> <rect x="478" y="150" width="222" height="38" rx="7" fill="#f1f5f9" stroke="#94a3b8"/> <text x="589" y="174" text-anchor="middle" font-size="11.5" fill="#64748b">mesh1: reviews (0 pods)</text> <text x="360" y="208" text-anchor="middle" font-size="11" fill="#b91c1c">Local hostname: local-only, and there is nothing local left.</text> </svg></div>

## 1.6 · Take over the local hostname too

**What we're doing:** make the app's own local hostname fail over as well, without touching the app. Real apps like productpage call `reviews.bookinfo.svc.cluster.local`, and you are not going to rewrite them.

**How:** one more label, `solo.io/service-takeover=true` (Solo Istio 1.27.2+). It redirects the local hostname onto the global endpoint set, so productpage's unchanged calls now reach mesh2. (`solo.io/service-scope=global-only` is the older single-label spelling of the same behaviour.)

**What you'll see:** the local hostname served cross-cluster by mesh2, and after a browser refresh the reviews panel back, now with **red stars** because mesh2's v3 is answering. That is cross-cluster traffic you can point at on the screen.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo label svc reviews solo.io/service-takeover=true --overwrite
sleep 10

echo "  ══ LOCAL hostname, taken over  ·  now served cross-cluster by mesh2 ══"
for i in $(seq 1 6); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]-[a-z0-9]+-[a-z0-9]+'
done | sort | uniq -c | awk '{printf "    %-34s ×%d\n", $2, $1}'
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo
echo "    refresh →  http://$GW:8080/productpage   (reviews back, red stars = mesh2's v3)"

<details>
<summary><strong>What changed (click to expand): the takeover label, and the local Service now resolves cross-cluster</strong></summary>

`solo.io/service-takeover=true` does not add a new object. It adds one label to the **same** autogen ServiceEntry from §1.4:

```yaml
metadata:
  name: autogen.bookinfo.reviews          # namespace: istio-system
  labels:
    solo.io/service-scope: global
    solo.io/service-takeover: "true"       # the only new field vs 1.4
```

The real effect is in the data plane. istiod reprograms **ztunnel** so the ordinary Kubernetes Service `reviews` (the `svc.cluster.local` name, VIP `10.96.37.233`) now has a remote endpoint, even though every local reviews pod is scaled to zero:

```
$ istioctl --context kind-mesh1 ztunnel-config services | grep reviews
bookinfo   reviews       10.96.37.233   None   1/1     # local Service still shows an endpoint...
bookinfo   reviews-v1    ...            None   0/0
bookinfo   reviews-v2    ...            None   0/0

$ istioctl --context kind-mesh1 ztunnel-config workloads | grep reviews
bookinfo   autogen.mesh2.bookinfo.reviews   ...   HBONE   mesh2   mesh2/192.168.97.160
```

That endpoint lives on **network mesh2**, reached through **mesh2's east-west gateway (192.168.97.160)** over HBONE. So productpage's unchanged call to `reviews.bookinfo.svc.cluster.local` now lands on mesh2, with no change to the app.

See it live:
```
kubectl --context kind-mesh1 -n istio-system get serviceentry autogen.bookinfo.reviews -o yaml
istioctl --context kind-mesh1 ztunnel-config services  | grep reviews
istioctl --context kind-mesh1 ztunnel-config workloads | grep reviews
```

</details>

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 250" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif"> <rect x="0" y="0" width="720" height="250" rx="10" fill="#f8fafc"/> <text x="360" y="27" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">Takeover: the local hostname now fails over to mesh2 as well</text> <defs> <marker id="g2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker> </defs> <!-- LANE 1: global hostname (unchanged) --> <rect x="20" y="58" width="120" height="38" rx="7" fill="#eef2ff" stroke="#6366f1"/> <text x="80" y="82" text-anchor="middle" font-size="11.5" fill="#312e81">caller (mesh1)</text> <line x1="140" y1="77" x2="198" y2="77" stroke="#16a34a" stroke-width="2" marker-end="url(#g2)"/> <rect x="198" y="58" width="222" height="38" rx="7" fill="#dbeafe" stroke="#60a5fa"/> <text x="309" y="82" text-anchor="middle" font-size="11" fill="#1e293b">reviews.bookinfo.mesh.internal</text> <text x="448" y="70" text-anchor="middle" font-size="9.5" fill="#166534">HBONE</text> <line x1="420" y1="77" x2="478" y2="77" stroke="#16a34a" stroke-width="2" marker-end="url(#g2)"/> <rect x="478" y="58" width="222" height="38" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="589" y="82" text-anchor="middle" font-size="11.5" font-weight="600" fill="#14532d">mesh2: reviews v3  ✓</text> <text x="360" y="116" text-anchor="middle" font-size="11" fill="#166534">Global hostname: still served from mesh2.</text> <!-- LANE 2: local hostname, NOW taken over to mesh2 --> <rect x="20" y="150" width="120" height="38" rx="7" fill="#eef2ff" stroke="#6366f1"/> <text x="80" y="174" text-anchor="middle" font-size="11.5" fill="#312e81">caller (mesh1)</text> <line x1="140" y1="169" x2="198" y2="169" stroke="#16a34a" stroke-width="2" marker-end="url(#g2)"/> <rect x="198" y="150" width="222" height="38" rx="7" fill="#dbeafe" stroke="#60a5fa"/> <text x="309" y="167" text-anchor="middle" font-size="10.5" fill="#1e293b">reviews...svc.cluster.local</text> <text x="309" y="181" text-anchor="middle" font-size="9" fill="#7c2d12">service-takeover=true</text> <text x="448" y="162" text-anchor="middle" font-size="9.5" fill="#166534">HBONE</text> <line x1="420" y1="169" x2="478" y2="169" stroke="#16a34a" stroke-width="2" marker-end="url(#g2)"/> <rect x="478" y="150" width="222" height="38" rx="7" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/> <text x="589" y="174" text-anchor="middle" font-size="11.5" font-weight="600" fill="#14532d">mesh2: reviews v3  ✓</text> <text x="360" y="208" text-anchor="middle" font-size="11" fill="#166534">Local hostname: redirected to mesh2, the app is unchanged (red stars in the browser).</text> </svg></div>

**Gloo UI moment.** In the Graph, the `demo-client` edge that was on mesh1's reviews has now crossed into **mesh2's** reviews. That is the cross-cluster hop, drawn live by the steady background traffic, no manual curling. (Make sure **"Idle Nodes" is OFF** in Graph Settings, or the mesh2 edge will stay on screen even after traffic leaves it.)

Now restore mesh1 and watch locality reassert. With Idle Nodes off, within ~30s the mesh2 edge drops and the graph edge returns to **mesh1** (and the productpage stars go black again). Traffic only crossed clusters during the outage. If the mesh2 edge lingers, Idle Nodes is still on. The `curl` counts below are the instant, authoritative truth.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
for v in v1 v2; do kubectl --context $CLUSTER1 -n bookinfo scale deploy reviews-$v --replicas=1; done
kubectl --context $CLUSTER1 -n bookinfo wait --for=condition=Ready pod -l app=reviews --timeout=120s
sleep 12
echo "  ══ Local endpoints back  ·  traffic returns to mesh1 (PreferNetwork) ══"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews.bookinfo.svc.cluster.local:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]-[a-z0-9]+-[a-z0-9]+'
done | sort | uniq -c | awk '{printf "    %-34s ×%d\n", $2, $1}'
echo "    no v3 = all mesh1  ·  cross-cluster only during the outage"

## 1.7 · The same gateway does traffic management too

**What we're doing:** before we leave the ingress, show it is a full L7 gateway, not just a front door.

**How:** two plain Gateway API objects on the same ingress from §1.3:

- an `HTTPRoute` that weights `/reviews` 80/20 across v1 and v2
- a policy that rate-limits `/productpage` to 2 requests a second

**What you'll see:** roughly an 80/20 split across the two review versions in the curl counts, and the productpage requests going 200, 200, then 429 once the limit bites.

These hit the ingress routes directly, so they show in the counts below, not in the productpage panel.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-canary, namespace: bookinfo }
spec:
  parentRefs: [{ name: bookinfo-gateway }]
  rules:
  - matches: [{ path: { type: PathPrefix, value: /reviews } }]
    backendRefs:
    - { name: reviews-v1, port: 9080, weight: 80 }
    - { name: reviews-v2, port: 9080, weight: 20 }
EOF
sleep 3
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo
echo "  ══ 20 requests through the ingress  ·  target 80/20 ══"
for i in $(seq 1 20); do curl -s "http://$GW:8080/reviews/0" | grep -oE 'reviews-v[0-9]'; done \
  | sort | uniq -c | awk '{printf "    %-11s %2d / 20   (%d%%)\n", $2, $1, $1*5}'

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: agentgateway.dev/v1alpha1
kind: AgentgatewayPolicy
metadata: { name: ingress-ratelimit, namespace: bookinfo }
spec:
  targetRefs:
  - { group: gateway.networking.k8s.io, kind: HTTPRoute, name: productpage }
  traffic:
    rateLimit:
      local:
      - requests: 2
        unit: Seconds
EOF
sleep 4
GW=$(kubectl --context $CLUSTER1 -n bookinfo get gateway bookinfo-gateway -o jsonpath='{.status.addresses[0].value}')
echo
echo "  ══ 8 rapid requests to /productpage  ·  limit 2/sec ══"
printf '    '; for i in $(seq 1 8); do curl -s -o /dev/null -w "%{http_code} " "http://$GW:8080/productpage"; done; echo
echo "    → first 2 pass (200), the rest limited (429)"

## 1.8 · L7 between services: the waypoint

**What we're doing:** add L7 control for traffic between services inside the mesh, not just at the edge.

**How:** ambient adds a **waypoint**, and on Solo Enterprise the waypoint is **agentgateway** (class `enterprise-agentgateway-waypoint`), not Envoy. Enrol the namespace onto it with one label, then attach an `HTTPRoute` whose parent is the reviews **Service**, so productpage's own calls to reviews are routed with no ingress involved.

**What you'll see:** every mesh call to reviews pinned to v2, agentgateway logging each request, and in the browser reviews staying solid **black** (v2) on every refresh.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 660 240" style="width:100%;max-width:1000px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="660" height="240" rx="10" fill="#f8fafc"/>
  <text x="330" y="26" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">A waypoint for service-to-service L7 (agentgateway), pinning reviews to v2</text>
  <defs>
    <marker id="mg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#16a34a"/></marker>
    <marker id="mx" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#94a3b8"/></marker>
  </defs>
  <rect x="20" y="48" width="620" height="176" rx="10" fill="#eff6ff" stroke="#3b82f6" stroke-width="2"/>
  <text x="330" y="68" text-anchor="middle" font-size="12" font-weight="700" fill="#1e3a8a">mesh1 · bookinfo namespace (ambient)</text>
  <rect x="42" y="112" width="120" height="46" rx="8" fill="#dbeafe" stroke="#60a5fa" stroke-width="1.5"/>
  <text x="102" y="140" text-anchor="middle" font-size="12" font-weight="600" fill="#1e293b">productpage</text>
  <line x1="162" y1="135" x2="226" y2="135" stroke="#16a34a" stroke-width="2" marker-end="url(#mg)"/>
  <rect x="228" y="104" width="200" height="64" rx="9" fill="#dcfce7" stroke="#16a34a" stroke-width="1.8"/>
  <text x="328" y="127" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">waypoint (agentgateway)</text>
  <text x="328" y="143" text-anchor="middle" font-size="9.5" fill="#166534">HTTPRoute on the reviews Service</text>
  <text x="328" y="156" text-anchor="middle" font-size="9.5" fill="#166534">in-mesh, no ingress</text>
  <line x1="428" y1="126" x2="478" y2="112" stroke="#16a34a" stroke-width="2" marker-end="url(#mg)"/>
  <text x="452" y="102" text-anchor="middle" font-size="9" fill="#166534">100%</text>
  <rect x="478" y="92" width="150" height="40" rx="8" fill="#dcfce7" stroke="#16a34a" stroke-width="1.5"/>
  <text x="553" y="117" text-anchor="middle" font-size="12" font-weight="600" fill="#14532d">reviews-v2  ✓</text>
  <line x1="428" y1="146" x2="478" y2="160" stroke="#94a3b8" stroke-width="1.6" stroke-dasharray="5 3" marker-end="url(#mx)"/>
  <text x="452" y="180" text-anchor="middle" font-size="9" fill="#64748b">0%</text>
  <rect x="478" y="150" width="150" height="40" rx="8" fill="#f1f5f9" stroke="#94a3b8"/>
  <text x="553" y="175" text-anchor="middle" font-size="12" fill="#64748b">reviews-v1  (idle)</text>
  <text x="330" y="214" text-anchor="middle" font-size="11" fill="#64748b">productpage's own calls to reviews now run through the waypoint; the route pins every request to v2 (black stars).</text>
</svg></div>

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# the waypoint: a Gateway of class enterprise-agentgateway-waypoint, then enrol the
# bookinfo NAMESPACE onto it (one waypoint fronts every service in the namespace)
kubectl --context $CLUSTER1 apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: bookinfo-waypoint
  namespace: bookinfo
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
kubectl --context $CLUSTER1 label namespace bookinfo istio.io/use-waypoint=bookinfo-waypoint --overwrite
kubectl --context $CLUSTER1 -n bookinfo rollout status deploy/bookinfo-waypoint --timeout=150s

# the agentgateway data plane the Gateway rendered (Deployment + Service + pod)
kubectl --context $CLUSTER1 -n bookinfo get deploy,svc,pod -l gateway.networking.k8s.io/gateway-name=bookinfo-waypoint

<details>
<summary><strong>What got created (click to expand): the waypoint the Gateway rendered</strong></summary>

You applied one `Gateway` (class `enterprise-agentgateway-waypoint`) and never wrote a Deployment, yet the agentgateway controller renders a full data plane from it. The `Deployment`, `Service`, `ServiceAccount` and `Pod` printed above are all rendered from that single Gateway; the pod runs the agentgateway image, listening on `15088` (mesh) and `15008` (HBONE).

```
$ kubectl -n bookinfo get gateway bookinfo-waypoint
NAME                CLASS                              PROGRAMMED   ADDRESS
bookinfo-waypoint   enterprise-agentgateway-waypoint   True         10.96.209.19
```

The `istio.io/use-waypoint=bookinfo-waypoint` namespace label is what makes ztunnel send every bookinfo service's traffic through this proxy. It is the same class of object as demo-3's petshop waypoint.

See it live:
```
kubectl --context kind-mesh1 -n bookinfo get gateway bookinfo-waypoint -o yaml
```

</details>

### Route at the waypoint (L7 HTTP routing)

With the waypoint now in front of every call to reviews, do the actual **L7 routing**. Attach an `HTTPRoute` whose `parentRef` is the reviews **Service** and pin all of productpage's traffic to **v2**.

Because the route's parent is a Service and not a `Gateway`, this is east-west, in-mesh routing (the Gateway API **GAMMA** pattern), so no ingress is involved. The same HTTPRoute shape you would put on an ingress now governs service-to-service calls.

**What you'll see:** all 8 mesh calls to reviews served by **v2**, and the agentgateway waypoint logging each request.

In [ ]:
: "${CLUSTER1:=kind-mesh1}" "${CLUSTER2:=kind-mesh2}" "${ISTIOCTL:=$HOME/.istioctl/bin/istioctl-1.30.3-solo}"
# route AT the waypoint: parentRef is the reviews SERVICE (GAMMA), pin every call to v2
kubectl --context $CLUSTER1 -n bookinfo apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: reviews-mesh-split, namespace: bookinfo }
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: reviews
      port: 9080
  rules:
    - backendRefs:
        - { name: reviews-v2, port: 9080, weight: 100 }
EOF
sleep 8
echo "--- 8 mesh calls to reviews (via the waypoint): all v2 ---"
for i in $(seq 1 8); do
  kubectl --context $CLUSTER1 -n bookinfo exec deploy/demo-client -- \
    curl -s -m 6 http://reviews:9080/reviews/0 2>/dev/null | grep -oE 'reviews-v[0-9]'
done | sort | uniq -c
# proof the waypoint carried it: agentgateway logs every request
kubectl --context $CLUSTER1 -n bookinfo logs deploy/bookinfo-waypoint --tail=3